# Rule-Based Evaluation Pipeline

In [33]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('Complaints.csv')
df = df[df['complaint_date'] != 'complaint_date'].copy()

orders_df = pd.read_csv('Orders.csv')
df['complaint_date'] = pd.to_datetime(df['complaint_date'], errors='coerce')
df = df.sort_values(by=['customer_id', 'complaint_date']).reset_index(drop=True)
df['prior_complaints'] = df.groupby('customer_id').cumcount()
order_counts = orders_df.groupby('customer_id').size().to_dict()
df['total_orders'] = df['customer_id'].map(order_counts).fillna(0)
df['complaint_ratio'] = (df['prior_complaints'] + 1) / df['total_orders'].apply(lambda x: 1 if x == 0 else x)
test_df = df.sample(frac=0.2).copy()

In [34]:
test_unlabeled = test_df[['complaint_id', 'complaint_text', 'customer_id', 'prior_complaints', 'complaint_ratio', 'total_orders']].copy()
test_unlabeled.sample(5)

,complaint_id,complaint_text,customer_id,prior_complaints,complaint_ratio,total_orders
113,CMP-239,i guess its fine but still refund pls,C-021,3,0.500,8
474,CMP-095,charging port loose,C-095,0,0.125,8
175,CMP-034,This doesn’t match my style,C-034,0,0.125,8
205,CMP-447,box had marks inside,C-039,7,1.000,8
372,CMP-467,box had scratches inside,C-073,4,0.625,8


In [35]:
def compute_fraud_features(row):
    text = str(row['complaint_text']).lower()
    score = 6
    classification = "Legitimate"
    
    if 'box' in text and any(w in text for w in ['random', 'broken', 'scratches', 'tampered', 'older', 'different']):
        score, classification = 95, 'Fraud'
    elif 'ordered' in text and ('got' in text or 'looks' in text or 'why' in text):
        score, classification = 93, 'Fraud'
    elif 'used' in text and ('already' in text or 'returning' in text or 'few' in text or 'event' in text or 'now' in text):
        score, classification = 85, 'Fraud'
    elif any(phrase in text for phrase in ['wrong item', 'wrong model', 'different from', 'not matching', 'match description']): 
        score, classification = 90, 'Fraud'
    elif any(phrase in text for phrase in ['changed my mind', 'dont like', "don't like", 'prefer', 'not my style', 'expected better', 'not what i expected', 'feels cheap', 'not worth']):
        score, classification = 18, 'Legitimate'
    else:
        words = text.split()
        if text.strip() in ['ok', 'bad product', 'not good', 'worst', 'bad', 'meh'] or (len(words) <= 3 and 'dead' not in text and 'not working' not in text):
            score, classification = 22, 'Legitimate'



    prior_complaints = row['prior_complaints']
    ratio = row['complaint_ratio']
    
    if prior_complaints >= 3:
        score += 15
        classification = "Fraud"
        
    if ratio > 0.5 and row['total_orders'] > 2:
        score += 10
        classification = "Fraud"

    if ratio <= 0.1 and prior_complaints == 0:
        score -= 5  
        
   
    score = max(0, min(100, score))
    if score >= 80:
        classification = "Fraud"
        
    return classification


test_unlabeled['Predicted_Fraud_Class'] = test_unlabeled.apply(compute_fraud_features, axis=1)
test_unlabeled.head()

,complaint_id,complaint_text,customer_id,prior_complaints,complaint_ratio,total_orders,Predicted_Fraud_Class
347,CMP-069,Earbuds volume very low,C-069,0,0.125,8,Legitimate
466,CMP-443,keyboard typing delay issue,C-093,3,0.500,8,Fraud
6,CMP-111,phone dead after update,C-002,1,0.250,8,Legitimate
157,CMP-030,Phone heats slightly during gaming,C-030,0,0.125,8,Legitimate
78,CMP-148,tablet ok but dont like it,C-015,2,0.375,8,Legitimate


In [36]:

accuracy = accuracy_score(test_df['fraud_classification'], test_unlabeled['Predicted_Fraud_Class'])

print(f" Accuracy: {accuracy * 100:.2f}%")
print("\nOverall Report:\n")
print(classification_report(test_df['fraud_classification'], test_unlabeled['Predicted_Fraud_Class']))

 Accuracy: 65.00%

Overall Report:

              precision    recall  f1-score   support

       Fraud       0.66      0.65      0.65        51
  Legitimate       0.64      0.65      0.65        49

    accuracy                           0.65       100
   macro avg       0.65      0.65      0.65       100
weighted avg       0.65      0.65      0.65       100

